In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/late_fusion/'
import pandas as pd

catalog = pd.read_csv(BASE_PATH + 'tracks_catalog.csv')

print("Catalog shape:", catalog.shape)
catalog.head()
#Load các object nhỏ (TF-IDF + scaler)
import joblib

tfidf_lyrics = joblib.load(BASE_PATH + 'tfidf_lyrics.pkl')
tfidf_metadata = joblib.load(BASE_PATH + 'tfidf_metadata.pkl')
scaler_audio = joblib.load(BASE_PATH + 'scaler_audio.pkl')

print("Loaded TF-IDF & scaler")

#Load similarity matrices lớn (.npz)
import numpy as np

sim_data = np.load(BASE_PATH + 'similarity_matrices.npz')

sim_lyrics = sim_data['sim_lyrics']
sim_audio = sim_data['sim_audio']
sim_metadata = sim_data['sim_metadata']

print(sim_lyrics.shape, sim_audio.shape, sim_metadata.shape)


Mounted at /content/drive
Catalog shape: (11256, 7)
Loaded TF-IDF & scaler
(11256, 11256) (11256, 11256) (11256, 11256)


In [ ]:
#Fusion content similarity (late fusion)
w_lyrics = 0.4
w_audio = 0.3
w_metadata = 0.3

sim_content = (
    w_lyrics * sim_lyrics +
    w_audio * sim_audio +
    w_metadata * sim_metadata
)

print("Content similarity ready:", sim_content.shape)


Content similarity ready: (11256, 11256)


In [ ]:
!pip install implicit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=938107 sha256=c4e21440a31496c0c7587fbcf0e12a0d751b1180376636be66a6f7302640bef3
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [ ]:
#Tạo mapping track_id ↔ index
track2idx = {tid: i for i, tid in enumerate(catalog['track_id'])}
idx2track = {i: tid for tid, i in track2idx.items()}
import implicit
#Load ALS model (CF)
model_pack = joblib.load('/content/drive/MyDrive/models/als_model.pkl')

print("ALS model loaded")


ALS model loaded


In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/datasets/data_train.csv')

print(train_df.head())


                                    user_id  play_count    rating  \
0  0002e94348b2543c6e6ccf408b0160d14064e46f           9  2.302585   
1  0002e94348b2543c6e6ccf408b0160d14064e46f           2  1.098612   
2  0002e94348b2543c6e6ccf408b0160d14064e46f           2  1.098612   
3  0002e94348b2543c6e6ccf408b0160d14064e46f           1  0.693147   
4  0002e94348b2543c6e6ccf408b0160d14064e46f           1  0.693147   

             track_id      artist_name                   title  \
0  TREXJJE128E0781B1D        Radiohead                    Just   
1  TRVUGOX128E0784629    Guns N' Roses    Don't Cry (Original)   
2  TROMKCG128F9320C09             Muse                Uprising   
3  TRGSVZA128F92CF92E      The Killers            Losing Touch   
4  TREQUZI128F92FB6C1  Thelonious Monk  Crepuscule With Nellie   

      artist_clean             title_clean  valence  year  ...  liveness  \
0        radiohead                    just   0.3580  1995  ...    0.0763   
1     guns n roses                do

In [ ]:
import numpy as np
import pickle
import gzip
import os
from google.colab import files

class HybridRecommender:
    def __init__(
        self,
        model_pack,
        train_df=None,
        sim_content=None,
        track2idx=None,
        alpha=0.7,
        cf_candidates=500
    ):
        """
        model_pack: dict (ALS model + mappings)
        train_df: user_id, track_id, play_count (chỉ cần khi init ban đầu)
        sim_content: fused similarity matrix (NxN)
        track2idx: mapping track_id -> index in sim_content
        """
        self.model = model_pack["model"]
        self.matrix = model_pack["matrix"]
        self.user2idx = model_pack["user2idx"]
        self.idx2song = model_pack["idx2song"]

        self.train_df = train_df
        self.sim_content = sim_content
        self.track2idx = track2idx
        if track2idx is not None:
            self.idx2track = {v: k for k, v in track2idx.items()}
        else:
            self.idx2track = None

        self.alpha = alpha
        self.cf_candidates = cf_candidates
        self.popularity = None  # Sẽ gán khi load hoặc init

        if train_df is not None:
            self.popularity = train_df.groupby('track_id')['play_count'].sum().sort_values(ascending=False)

    # ---------- CF ----------
    def _cf_candidates(self, user_id):
        if user_id not in self.user2idx:
            return None, None

        uidx = self.user2idx[user_id]
        ids, scores = self.model.recommend(
            uidx,
            self.matrix[uidx],
            N=self.cf_candidates,
            filter_already_liked_items=True
        )

        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        return ids, scores

    # ---------- Content ----------
    def _content_scores(self, user_id):
        if self.train_df is None:
            # Khi load, dùng fallback nếu cần
            return None

        user_hist = self.train_df[self.train_df.user_id == user_id] if self.train_df is not None else None
        if user_hist is None or user_hist.empty:
            return None

        scores = np.zeros(self.sim_content.shape[0], dtype=np.float32)

        for _, row in user_hist.iterrows():
            tid = row.track_id
            if tid not in self.track2idx:
                continue
            idx = self.track2idx[tid]
            scores += self.sim_content[idx] * row.play_count

        scores /= (scores.max() + 1e-8)
        return scores

    # ---------- Popularity for cold users ----------
    def _popularity_recs(self, topK=10):
        return list(self.popularity.index[:topK])

    # ---------- PUBLIC API ----------
    def recommend(self, user_id, topK=10):
        content_scores = self._content_scores(user_id)
        if content_scores is None:
            # User cold start: fallback to popularity
            return self._popularity_recs(topK)

        ids, cf_scores = self._cf_candidates(user_id)

        # Get content candidates
        content_idx = np.argsort(-content_scores)[:self.cf_candidates]
        content_tracks = [self.idx2track[i] for i in content_idx]
        content_scores_dict = {self.idx2track[i]: content_scores[i] for i in content_idx}

        # Get CF candidates if available
        if ids is not None:
            cf_tracks = [self.idx2song[i] for i in ids]
            cf_scores_dict = dict(zip(cf_tracks, cf_scores))
            all_candidates = set(cf_tracks) | set(content_tracks)
        else:
            # If no CF (though unlikely if content exists), use content only
            cf_scores_dict = {}
            all_candidates = set(content_tracks)

        # Compute final scores
        final_scores = []
        for track in all_candidates:
            c_s = content_scores[self.track2idx[track]] if track in self.track2idx else 0
            cf_s = cf_scores_dict.get(track, 0)
            if track in cf_scores_dict:
                # Normal item: weighted hybrid
                score = self.alpha * cf_s + (1 - self.alpha) * c_s
            else:
                # Item cold start: content only
                score = c_s
            final_scores.append((track, score))

        # Rank and return topK
        ranked = sorted(final_scores, key=lambda x: x[1], reverse=True)
        return [t for t, _ in ranked[:topK]]

    @classmethod
    def load(cls, load_path="models/hybrid_recommender_v1.pkl.gz"):
        """
        Load lại recommender từ file pickle (hỗ trợ gzip).
        """
        if not os.path.exists(load_path):
            raise FileNotFoundError(f"Không tìm thấy file: {load_path}")

        if load_path.endswith('.gz'):
            with gzip.open(load_path, "rb") as f:
                data = pickle.load(f)
        else:
            with open(load_path, "rb") as f:
                data = pickle.load(f)

        # Tái tạo instance
        recommender = cls(
            model_pack={
                "model": data["model"],
                "matrix": data["matrix"],
                "user2idx": data["user2idx"],
                "idx2song": data["idx2song"],
            },
            train_df=None,  # Không cần train_df đầy đủ
            sim_content=data["sim_content"],
            track2idx=data["track2idx"],
            alpha=data["alpha"],
            cf_candidates=data["cf_candidates"]
        )

        # Gán lại các thuộc tính
        recommender.idx2track = data["idx2track"]
        recommender.popularity = data["popularity"]  # Series popularity đã lưu

        print(f"HybridRecommender đã được load từ: {load_path}")
        return recommender

# ------------------- PHẦN LƯU TRONG COLAB -------------------
os.makedirs("models", exist_ok=True)
save_path = "models/hybrid_recommender_v1.pkl.gz"

recommender = HybridRecommender(
    model_pack=model_pack,
    train_df=train_df,
    sim_content=sim_content,
    track2idx=track2idx,
    alpha=0.7,
    cf_candidates=500
)

save_data = {
    "model": recommender.model,
    "matrix": recommender.matrix,
    "user2idx": recommender.user2idx,
    "idx2song": recommender.idx2song,
    "idx2track": recommender.idx2track,
    "track2idx": recommender.track2idx,
    "sim_content": recommender.sim_content,
    "popularity": recommender.popularity,
    "alpha": recommender.alpha,
    "cf_candidates": recommender.cf_candidates,
}

with gzip.open(save_path, "wb") as f:
    pickle.dump(save_data, f)

print(f"Đã lưu model nén thành công tại: {save_path}")

# Tải về máy local ngay lập tức
files.download(save_path)


Đã lưu model nén thành công tại: models/hybrid_recommender_v1.pkl.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
hybrid_rec = HybridRecommender(
    model_pack=model_pack,
    train_df=train_df,
    sim_content=sim_content,
    track2idx=track2idx,
    alpha=0.7,
    cf_candidates=500
)


In [ ]:
user = train_df.user_id.unique()[10]

rec_ids = hybrid_rec.recommend(user, topK=10)

catalog[catalog.track_id.isin(rec_ids)]


,track_id,title,artist_name,year,popularity,decade,has_lyrics
701,TRWMLSX12903CCD317,Never Gonna Give You Up,The Black Keys,2010,49,2010s,True
706,TREVCQU12903CCD318,These Days,The Black Keys,2010,44,2010s,True
721,TRPFJPH128F427A71D,Psychotic Girl,The Black Keys,2008,49,2000s,True
725,TROEGQQ12903CCD316,Unknown Brother,The Black Keys,2010,44,2010s,True
738,TRNBJUK12903CD9C89,Meet Me In the City,The Black Keys,2006,53,2000s,True
745,TRKSEEY12903CCD312,Ten Cent Pistol,The Black Keys,2010,50,2010s,True
751,TRLYQJU12903CCD315,Im Not The One,The Black Keys,2010,46,2010s,True
1514,TRSYPTC12903CB4521,Act Nice and Gentle,The Black Keys,2004,45,2000s,True
1521,TRSVTRT12903CB450E,The Lengths,The Black Keys,2004,50,2000s,True
1968,TRPUGEO12903CA9D27,Strange Desire,The Black Keys,2006,39,2000s,False
